In [78]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, f_classif, mutual_info_classif,
    RFE, SelectFromModel
)
from sklearn.metrics import roc_auc_score, classification_report
from functools import partial

In [3]:
# Переменная для генерации случайных чисел
RANDOM_STATE = 42
# Схема кросс-валидации и генерация случайнных чисел
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# Настройка отображения до 40 столбцов
pd.set_option('display.max_columns', 40)

"""Есть набор данных"""
data = load_breast_cancer(as_frame=True)
"""Он был раздёлен на обучающую выборку и целевую переменную для предсказания. Переменные X_train"""
X_original = data.data.copy()
y = data.target.copy()

# Создание случайных чисел
rng = np.random.default_rng(RANDOM_STATE)
# Копирование всех столбцов без целевой переменной
X = X_original.copy()
# Добавление сгенерированных случайнных чисел
for j in range(8):
    # Создание столбца и заполнение по нормальному распределению Гаусса его значением в количестве длинны тренировачных данных
    X[f'noise_{j + 1}'] = rng.normal(size=len(X))

# Создание дополнительных столбцов
X['duplicate_mean_radius'] = X['mean radius'] + rng.normal(0, 0.02, len(X))
X['duplicate_mean_texture'] = X['mean texture'] + rng.normal(0, 0.02, len(X))
X['constant_feature'] = 1.0
# Разделение на тренировочные и валидационные данные
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)


In [4]:
# resume
print("☕")
"""
Признаки это столбцы
wzxdfccadsdshrqepautrjiuth
Строка это объёкт, а у объекта есть признак или по сути свойство с какимто значением.
Синонимы, которые вы встретите: feature, признак, переменная, колонка, атрибут, фактор. В коде это буквально X.columns

Среди столбцов есть шум по типу "noise_1" и constant_feature, которые бесмысленны.
По заданию и нужно выявить только необходимые признаки для правильного обучения с помощью методов, а не вручную.
duplicate_mean_radius, duplicate_mean_texture — 2 почти-дубликата реальных признаков (те же значения + маленький шум).
constant_feature — 1 константа (везде 1.0, никакой информации не несёт).
"""
print(X_train.shape)          # (426, 41) — 426 строк, 41 столбец (признак)
print(list(X_train.columns))  # список всех признаков
X_train.head(3)               # первые 3 строки таблицы
print("Оригинальный набор данных")
print(data.data.shape[0], data.data.shape[1])
data.data.head()
"""🔳Терминология хуже дотерской...
Опухоль — это объект (пациент), а y — его класс (злокачественная / доброкачественная).
Класс — это одно из возможных значений целевой переменной y.
задача бинарной классификации — это значит классов для предсказания ровно два.
Если бы y был, скажем, {'кошка', 'собака', 'птица'} — классов было бы три.
Если бы y был числом (размер опухоли в мм) — это была бы не классификация, а регрессия, и понятия класс там нет.
Классы несбалансированны это значит
что количество объектов со значением целевой переменной например 1 не равно количество объектов со значением целевой переменной 0

Стоит сравнивать модели по метрике ROC-AUC если классы несбалансированны. Если больных людей 30%, а модель всех считает здоровыми, то её аккуратность
будет 70%, потому что именно 70% здоровы, однако это не означает, что новую группу людей модель сможет правильно классифицировать с точностью 70%.

Очистка данных это очищение данных от неинформативных элементов, а подбор признаков это взятие наиболее полезных признаков среди информативных.

Методы подбора признаков
filter - не трогает модель. Быстрый, но менее точный.
wrapper - точный. Но медленее. Обучает модель.
embended - баланс точности и скорости.
Фолд это определённое разделение или разбиение данных
"""
print("🔍⚡")
# С методом filter были убраны шумовые признаки
columns_with_noises = X_train.filter(like="noise")
X_train_clean = X_train.drop(columns=columns_with_noises)
print(X_train_clean.head(4))
# Количество колонок
print(X_train_clean.shape)
"""📝"""

☕
(426, 41)
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'noise_1', 'noise_2', 'noise_3', 'noise_4', 'noise_5', 'noise_6', 'noise_7', 'noise_8', 'duplicate_mean_radius', 'duplicate_mean_texture', 'constant_feature']
Оригинальный набор данных
569 30
🔍⚡
     mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
517        19.89         20.26          130.50     1214.0          0.10370   
287        12.89         13.12           81.89      515.9    

'📝'

In [30]:
# Матрица кореляций но все значения по модулю
corr = X_train_clean.corr().abs()
print(corr)

                         mean radius  mean texture  mean perimeter  mean area  \
mean radius                 1.000000      0.326956        0.997875   0.988667   
mean texture                0.326956      1.000000        0.329590   0.321316   
mean perimeter              0.997875      0.329590        1.000000   0.987645   
mean area                   0.988667      0.321316        0.987645   1.000000   
mean smoothness             0.163974      0.042455        0.201531   0.177509   
mean compactness            0.508447      0.213223        0.559070   0.504337   
mean concavity              0.662470      0.276623        0.702194   0.671188   
mean concave points         0.821610      0.272073        0.850007   0.823343   
mean symmetry               0.142776      0.046871        0.178596   0.146884   
mean fractal dimension      0.321006      0.093514        0.270415   0.286907   
radius error                0.683935      0.246064        0.693439   0.734196   
texture error               

In [35]:
# Стандартное отклонение std . Остаются только те столбцы, чьё стандартное отклонение больше нуля
# Таким образом были удалены столбцы, которые имеют одинаковое значение во всех строках
wtf = X_train.loc[:, # Выбираются все колонки
    X_train.std() > 0] # У которых стандартное отклонение больше 0
print(wtf.shape)

(426, 40)


In [32]:
# Верхний треугольник. Всё значения выше диагонали
# Выбраны все значения где
upper = corr.where(
    # Они соответствуют
    np.triu(
        # Единичная матрица размера кореляционной, но вместо единицы или нуля стоит True или False
        np.ones(corr.shape).astype(bool),
        # не считая позицию k.
        k=1
    )
)
print(upper)

                         mean radius  mean texture  mean perimeter  mean area  \
mean radius                      NaN      0.326956        0.997875   0.988667   
mean texture                     NaN           NaN        0.329590   0.321316   
mean perimeter                   NaN           NaN             NaN   0.987645   
mean area                        NaN           NaN             NaN        NaN   
mean smoothness                  NaN           NaN             NaN        NaN   
mean compactness                 NaN           NaN             NaN        NaN   
mean concavity                   NaN           NaN             NaN        NaN   
mean concave points              NaN           NaN             NaN        NaN   
mean symmetry                    NaN           NaN             NaN        NaN   
mean fractal dimension           NaN           NaN             NaN        NaN   
radius error                     NaN           NaN             NaN        NaN   
texture error               

In [33]:
# Повторение функции any
print(any([False, False, False, False, False, False, False, True]))
print(any([False, False, False, False, False, False, False]))

True
False


In [40]:
# сохраняются все названия колонок
X_train_no_dublicates = [column for column in upper.columns
    # Если среди всех значений каждой колонки выше 0.95
    if (any(upper[column]) > 0.95)]
print(X_train_no_dublicates)
print(len(X_train_no_dublicates))

['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'duplicate_mean_radius', 'duplicate_mean_texture', 'constant_feature']
33


In [48]:
# Все лишние колонки
stup_x = [col for col in X_train.columns if not(  col in X_train_no_dublicates  )]
print(stup_x, len(stup_x), "\n⚡")
# Нужные колонки
stup_x = [col for col in X_train.columns if col in X_train_no_dublicates]
print(stup_x, len(stup_x))

['noise_1', 'noise_2', 'noise_3', 'noise_4', 'noise_5', 'noise_6', 'noise_7', 'noise_8'] 8 
⚡
['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness', 'mean compactness', 'mean concavity', 'mean concave points', 'mean symmetry', 'mean fractal dimension', 'radius error', 'texture error', 'perimeter error', 'area error', 'smoothness error', 'compactness error', 'concavity error', 'concave points error', 'symmetry error', 'fractal dimension error', 'worst radius', 'worst texture', 'worst perimeter', 'worst area', 'worst smoothness', 'worst compactness', 'worst concavity', 'worst concave points', 'worst symmetry', 'worst fractal dimension', 'duplicate_mean_radius', 'duplicate_mean_texture', 'constant_feature'] 33


In [58]:
# Наконец подбор признаков с помощью разновидности метода filter под названием SelectKBest

method_подбора_признаков_filter = SelectKBest(
    # Функция метода статистики для оценки каждого признака отдельно при помощи нахождения отличий между классами
    score_func=f_classif,
    # Составить топ столько-то признаков
    k=10
)
method_подбора_признаков_filter

,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...0021F07CB3880>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",10


In [60]:
# Список действий PiPeLiNe
task_list = Pipeline([
    # Стандартный параметр для масштабирования
    (  "scale", StandardScaler()  ),
    # Выбор самого метода
    (  "select", method_подбора_признаков_filter  ),
    (  "model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)  )
])
task_list

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scale', ...), ('select', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...0021F07CB3880>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",10
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000


In [63]:
# Указанное в задание разделение по данным
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Кросс валидация и сбор необходимой статистики по данным с помощью выбранных параметров
scores = cross_val_score(
    # Указывается pipeline или список действий
    task_list,
    X_train, y_train,
    # Фолды или настроенное разделение данных
    cv=CV,
    # Характеристика по которой будет оцениваться качество модели
    scoring='roc_auc'
)

D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [40] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [40] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning

In [68]:
# Превращение списка словарей в таблицу
список_словарей = ({"человека": 2}, {"человека": 2, "педалей": 3}, {"человека": 2, "колёса": 4})
таблица = pd.DataFrame(список_словарей)
таблица

,человека,педалей,колёса
0,2,NaN,NaN
1,2,3.0,NaN
2,2,NaN,4.0


In [79]:
# Решение блин
k = [5, 10, 15, 20, 30]

results = []

for number_k in k:
    method_подбора_признаков_filter = SelectKBest(
        # Функция метода статистики для оценки каждого признака отдельно при помощи нахождения отличий между классами
        score_func=f_classif,
        # Составить топ столько-то признаков
        k=number_k
    )
    
    task_list = Pipeline([
        # Стандартный параметр для масштабирования
        (  "scale", StandardScaler()  ),
        # Выбор самого метода
        (  "select", method_подбора_признаков_filter  ),
        (  "model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)  )
    ])

    scores = cross_val_score(
        # Указывается pipeline или список действий
        task_list,
        X_train, y_train,
        # Фолды или настроенное разделение данных
        cv=CV,
        # Характеристика по которой будет оцениваться качество модели
        scoring='roc_auc'
    )
    round(scores.mean(), 4)

    result = ["ANOVA",
        "k = " + str(number_k),
        str(round(scores.mean(), 4)),
        str(round(scores.std(), 4))
        ]
    
    results.append(result)

    # Второй метод

    method_подбора_признаков_filter = SelectKBest(
        # Функция метода статистики для оценки каждого признака отдельно при помощи нахождения отличий между классами
        score_func=partial(mutual_info_classif, random_state=RANDOM_STATE),
        # Составить топ столько-то признаков
        k=number_k
    )
    
    task_list = Pipeline([
        # Стандартный параметр для масштабирования
        (  "scale", StandardScaler()  ),
        # Выбор самого метода
        (  "select", method_подбора_признаков_filter  ),
        (  "model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)  )
    ])

    scores = cross_val_score(
        # Указывается pipeline или список действий
        task_list,
        X_train, y_train,
        # Фолды или настроенное разделение данных
        cv=CV,
        # Характеристика по которой будет оцениваться качество модели
        scoring='roc_auc'
    )
    round(scores.mean(), 4)

    result = ["MI",
        "k = " + str(number_k),
        str(round(scores.mean(), 4)),
        str(round(scores.std(), 4))
        ]
    
    results.append(result)

print(results)
print(pd.DataFrame(results))

D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [40] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [40] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
D:\python_programming\for_git_repositories\work_folder\STUDY\practice10_отбор_признаков\.venv\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning

[['ANOVA', 'k = 5', '0.9864'], ['MI', 'k = 5', '0.9872'], ['ANOVA', 'k = 10', '0.9865'], ['MI', 'k = 10', '0.9869'], ['ANOVA', 'k = 15', '0.9913'], ['MI', 'k = 15', '0.9906'], ['ANOVA', 'k = 20', '0.9965'], ['MI', 'k = 20', '0.996'], ['ANOVA', 'k = 30', '0.996'], ['MI', 'k = 30', '0.9958']]
       0       1       2
0  ANOVA   k = 5  0.9864
1     MI   k = 5  0.9872
2  ANOVA  k = 10  0.9865
3     MI  k = 10  0.9869
4  ANOVA  k = 15  0.9913
5     MI  k = 15  0.9906
6  ANOVA  k = 20  0.9965
7     MI  k = 20   0.996
8  ANOVA  k = 30   0.996
9     MI  k = 30  0.9958
